# Part 3: NLP and Sequence Modeling Mini Project
## Customer Support Sentiment Classification

**Dataset:** Customer Support Text Classification (1500 records, 3 classes: positive / neutral / negative)  
**Goal:** Build a complete NLP pipeline — from raw text to a trained LSTM model — and understand why sequence modeling matters.

---

## 0. Setup — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import re, warnings, json, os

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120

COLORS  = {'positive':'#2ecc71', 'neutral':'#3498db', 'negative':'#e74c3c'}
PALETTE = ['#2ecc71', '#3498db', '#e74c3c']

print('Libraries loaded successfully')

---
## Task 1: Dataset Understanding

We load the customer support text classification dataset and examine its structure, class distribution, and key statistics.

In [ ]:
df = pd.read_csv('customer_support_text_classification.csv')

print('=' * 55)
print('DATASET OVERVIEW')
print('=' * 55)
print(f'Total records   : {len(df):,}')
print(f'Columns         : {list(df.columns)}')
print(f'Missing values  : {df.isnull().sum().sum()}')
print()
print('--- Class Distribution ---')
print(df['sentiment_label'].value_counts())
print()
print('--- Channel Distribution ---')
print(df['channel'].value_counts())
print()
print('--- Word Count Stats ---')
print(df['word_count'].describe())
print()
print('--- Sample Records ---')
df[['customer_message','sentiment_label','channel']].head(6)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Dataset Overview - Customer Support Text Classification',
             fontsize=15, fontweight='bold', y=1.01)

label_counts = df['sentiment_label'].value_counts()
bars = axes[0].bar(label_counts.index, label_counts.values,
                   color=[COLORS[l] for l in label_counts.index],
                   edgecolor='white', linewidth=1.2)
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].set_xlabel('Sentiment Label'); axes[0].set_ylabel('Count')
for bar, v in zip(bars, label_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 str(v), ha='center', fontsize=11)
axes[0].set_ylim(0, label_counts.max()+60)

chan = df['channel'].value_counts()
wedge_colors = ['#9b59b6','#1abc9c','#e67e22','#e74c3c','#3498db']
axes[1].pie(chan.values, labels=chan.index, autopct='%1.1f%%',
            colors=wedge_colors, startangle=140,
            wedgeprops={'edgecolor':'white','linewidth':1.2})
axes[1].set_title('Channel Distribution', fontsize=13)

df['text_length'] = df['customer_message'].apply(len)
for lbl in ['positive','neutral','negative']:
    sub = df[df['sentiment_label']==lbl]['word_count']
    axes[2].hist(sub, bins=20, alpha=0.6, label=lbl, color=COLORS[lbl], edgecolor='white')
axes[2].set_title('Word Count Distribution by Sentiment', fontsize=13)
axes[2].set_xlabel('Word Count'); axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.savefig('results/dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:**
- The dataset is **well-balanced** across all three sentiment classes (~33% each) — no class-imbalance correction is needed.
- Tickets arrive from five channels (email, social, phone, chat, app) in roughly equal proportions.
- Word counts are concentrated between 7-18 words per message. Short, structured customer support text.
  This is why even TF-IDF performs well here, but sequence models are essential for longer, noisier real-world text.

---
## Task 2: Text Preprocessing

Raw text contains noise: punctuation, numbers, mixed casing, stopwords. We clean the text systematically.

**Steps:**
1. **Lowercasing** - removes case sensitivity
2. **Remove special characters and digits** - keeps only letters
3. **Tokenization** - splits text into words
4. **Stopword removal** - drops words like 'the', 'is', 'my' that carry little meaning

In [ ]:
STOPWORDS = {
    'i','me','my','myself','we','our','ours','ourselves','you','your',
    'yours','yourself','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs',
    'themselves','what','which','who','whom','this','that','these',
    'those','am','is','are','was','were','be','been','being',
    'have','has','had','having','do','does','did','doing','a','an',
    'the','and','but','if','or','because','as','until','while','of',
    'at','by','for','with','about','against','between','into','through',
    'during','before','after','above','below','to','from','up','down',
    'in','out','on','off','over','under','again','further','then','once',
    'here','there','when','where','why','how','all','both','each',
    'few','more','most','other','some','such','no','nor','not','only',
    'own','same','so','than','too','very','s','t','can','will','just',
    'should','now','ll','m','re','ve','y'
}

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 1]
    return ' '.join(tokens)

df['clean_text'] = df['customer_message'].apply(preprocess)

print('Before vs After Preprocessing:')
for _, row in df[['customer_message','clean_text','sentiment_label']].head(4).iterrows():
    print(f'\n[{row.sentiment_label.upper()}]')
    print(f'  ORIGINAL : {row.customer_message}')
    print(f'  CLEANED  : {row.clean_text}')

**Observation:** Ticket numbers, stopwords, and punctuation are removed. Only sentiment-bearing words remain.
For example, a negative review gets reduced to key words like 'refund pending frustrating experience'.

---
## Task 3: Text Vectorization

> **Why must text be converted to numbers?**
> Machine learning models work exclusively with numbers. Raw text strings have no mathematical meaning.
> We convert text to numerical vectors so the model can compute distances, gradients, and learned patterns.

| Method | How it works | Best for |
|---|---|---|
| **Bag of Words (BoW)** | Count word occurrences | Simple, fast baseline |
| **TF-IDF** | Weight words by uniqueness across corpus | Better signal-to-noise |
| **Tokenizer Sequences** | Map each word to an integer ID (preserves order) | LSTM / RNN models |

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y  = le.fit_transform(df['sentiment_label'])
print('Label encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

X_text = df['clean_text']
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X_text, y, test_size=0.20, random_state=42, stratify=y)
print(f'Training samples: {len(X_train_txt)} | Test samples: {len(X_test_txt)}')

# TF-IDF
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train_txt)
X_test_tfidf  = tfidf.transform(X_test_txt)
print(f'TF-IDF matrix shape : {X_train_tfidf.shape}')

# Bag of Words
bow = CountVectorizer(max_features=3000)
X_train_bow = bow.fit_transform(X_train_txt)
X_test_bow  = bow.transform(X_test_txt)
print(f'BoW matrix shape    : {X_train_bow.shape}')

# Show top TF-IDF features
sample_vec = tfidf.transform([X_train_txt.iloc[0]])
top_feats = sorted(zip(tfidf.get_feature_names_out(), sample_vec.toarray()[0]),
                   key=lambda x: x[1], reverse=True)[:8]
print(f'\nTop TF-IDF features for: "{X_train_txt.iloc[0]}"')
for word, score in top_feats:
    if score > 0:
        print(f'  {word:<30} score={score:.4f}')

**Key insight:** Each message becomes a 3000-dimensional vector. TF-IDF down-weights generic words
shared across all messages and up-weights distinctive sentiment words like 'frustrated', 'excellent', 'pending'.

---
## Task 4: Baseline Models

We build two classical ML baselines:
1. **Logistic Regression + TF-IDF** - strong linear classifier on sparse TF-IDF features
2. **Naive Bayes + Bag of Words** - probabilistic classifier, extremely fast to train

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, y_pred_lr)
lr_f1  = f1_score(y_test, y_pred_lr, average='macro')

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)
nb_acc = accuracy_score(y_test, y_pred_nb)
nb_f1  = f1_score(y_test, y_pred_nb, average='macro')

class_names = le.classes_

print('LOGISTIC REGRESSION (TF-IDF)')
print(f'  Accuracy : {lr_acc:.4f}  |  Macro F1 : {lr_f1:.4f}')
print(classification_report(y_test, y_pred_lr, target_names=class_names))

print('NAIVE BAYES (Bag of Words)')
print(f'  Accuracy : {nb_acc:.4f}  |  Macro F1 : {nb_f1:.4f}')
print(classification_report(y_test, y_pred_nb, target_names=class_names))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Baseline Model Confusion Matrices', fontsize=14, fontweight='bold')

for ax, cm_data, title in zip(
        axes,
        [confusion_matrix(y_test, y_pred_lr), confusion_matrix(y_test, y_pred_nb)],
        ['Logistic Regression (TF-IDF)', 'Naive Bayes (BoW)']):
    sns.heatmap(cm_data, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax,
                linewidths=0.5, cbar=False)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('results/baseline_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:**
Both classical models achieve perfect performance on this synthetic dataset because the language patterns are
deliberately clean and consistent. On real-world data, models typically achieve 70-85% accuracy.
The baseline validates our preprocessing pipeline and sets the performance ceiling for comparison with LSTM.

---
## Task 5: Sequence Model — LSTM

**Why sequence models?**
TF-IDF treats each word as independent. But meaning often depends on word *order*.
'not good' is very different from 'good', but TF-IDF cannot distinguish them after stopword removal.
LSTMs process words in order, maintaining a hidden state that carries forward context.

### LSTM Architecture

```
Input Text
   -> [Keras Tokenizer]    integer sequence (max_len=30, padded)
   -> [Embedding(5000,64)] dense word vector per token
   -> [SpatialDropout1D]   regularization on embeddings
   -> [LSTM(64)]           sequential context + memory cells
   -> [Dense(32, ReLU)]    high-level feature extraction
   -> [Dropout(0.3)]       prevent overfitting
   -> [Dense(3, Softmax)]  class probabilities (neg/neu/pos)
```

**Loss function:** Categorical Cross-Entropy  
**Optimiser:** Adam  
**Metric:** Accuracy + Macro F1

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping

MAX_VOCAB  = 5000
MAX_LEN    = 30
EMBED_DIM  = 64
BATCH_SIZE = 32
EPOCHS     = 20

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_txt)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_txt),
                             maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test_txt),
                             maxlen=MAX_LEN, padding='post', truncating='post')

print(f'Vocab size  : {MAX_VOCAB}')
print(f'Max seq len : {MAX_LEN}')
print(f'Train shape : {X_train_seq.shape}')
print(f'Test shape  : {X_test_seq.shape}')
print(f'\nSample original   : {X_train_txt.iloc[0]}')
print(f'Sample encoded    : {X_train_seq[0]}')

In [ ]:
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=3)
y_test_cat  = tf.keras.utils.to_categorical(y_test,  num_classes=3)

tf.random.set_seed(42)
model = Sequential([
    Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='LSTM_Sentiment_Classifier')

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
es = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=0)

history = model.fit(
    X_train_seq, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,
    callbacks=[es],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=14, fontweight='bold')
epochs_ran = range(1, len(history.history['accuracy'])+1)

axes[0].plot(epochs_ran, history.history['accuracy'],     'b-o', label='Train Acc', ms=5)
axes[0].plot(epochs_ran, history.history['val_accuracy'], 'r-o', label='Val Acc',   ms=5)
axes[0].set_title('Accuracy per Epoch'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history.history['loss'],     'b-o', label='Train Loss', ms=5)
axes[1].plot(epochs_ran, history.history['val_loss'], 'r-o', label='Val Loss',   ms=5)
axes[1].set_title('Loss per Epoch'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
lstm_loss, lstm_acc = model.evaluate(X_test_seq, y_test_cat, verbose=0)
y_pred_lstm = np.argmax(model.predict(X_test_seq, verbose=0), axis=1)
lstm_f1 = f1_score(y_test, y_pred_lstm, average='macro')

print(f'LSTM Test Accuracy : {lstm_acc:.4f}')
print(f'LSTM Macro F1      : {lstm_f1:.4f}')
print()
print(classification_report(y_test, y_pred_lstm, target_names=class_names))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_lstm), annot=True, fmt='d',
            cmap='Greens', xticklabels=class_names, yticklabels=class_names,
            ax=ax, linewidths=0.5, cbar=False)
ax.set_title('LSTM Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('results/lstm_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Model Comparison

In [ ]:
models_names = ['Logistic Regression\n(TF-IDF)', 'Naive Bayes\n(BoW)', 'LSTM\n(Sequences)']
accs = [lr_acc, nb_acc, lstm_acc]
f1s  = [lr_f1,  nb_f1,  lstm_f1]

x = np.arange(len(models_names)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - w/2, accs, w, label='Accuracy', color='#3498db', edgecolor='white')
b2 = ax.bar(x + w/2, f1s,  w, label='Macro F1', color='#e67e22', edgecolor='white')
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(models_names, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Comparison - Accuracy & Macro F1', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

results_df = pd.DataFrame({
    'Model'    : models_names,
    'Accuracy' : [f'{a:.4f}' for a in accs],
    'Macro F1' : [f'{f:.4f}' for f in f1s]
})
print(results_df.to_string(index=False))

**Interpretation:**
All three models achieve perfect performance on this synthetic dataset due to clean, template-based language.
On real-world noisy data, LSTM typically outperforms classical models by capturing word order and context
(e.g., 'not satisfied' vs 'satisfied') that TF-IDF collapses into the same features after preprocessing.

## Sample Predictions

In [ ]:
label_map   = {0:'negative', 1:'neutral', 2:'positive'}
X_test_list = X_test_txt.reset_index(drop=True)

print('LSTM SAMPLE PREDICTIONS')
print('=' * 65)
for i in range(10):
    true_lbl = label_map[y_test[i]]
    pred_lbl = label_map[y_pred_lstm[i]]
    mark = 'CORRECT' if true_lbl == pred_lbl else 'WRONG'
    print(f'[{mark}] {X_test_list[i]}')
    print(f'  True: {true_lbl:<12} Predicted: {pred_lbl}')
    print()

---
## Task 6: Attention and Transformer Reflection

### 6.1 — Why RNNs Struggle with Long-Term Dependencies

A basic RNN processes a sentence word by word, updating a single **hidden state** at each step.
The key problem is the **vanishing gradient problem**:

- During training, gradients flow backwards through each time step.
- With long sequences (100+ words), the gradient signal becomes exponentially smaller.
- By the time it reaches the first words, the gradient is essentially zero.
- The model **forgets** early context when making predictions about later parts.

**Example:**
> *'The customer, who called three times and waited over two hours on hold, was ultimately not satisfied.'*
> A plain RNN may forget 'not' by the time it reaches 'satisfied'.

---

### 6.2 — How LSTMs Solve Memory Problems

LSTMs introduce a **cell state** — a dedicated memory lane through the sequence.
Three learnable gates control this memory:

| Gate | Role |
|---|---|
| **Forget gate** | Decides what to erase from memory |
| **Input gate** | Decides what new information to write |
| **Output gate** | Decides what memory to expose as output |

Because the cell state can carry information unchanged across hundreds of steps, LSTMs can
remember context from early in the sequence. Our model used LSTM(64) — 64 memory units,
each independently gating its own cell state.

---

### 6.3 — What Attention Solves in Seq-to-Seq Tasks

Even LSTMs compress the entire source sentence into one fixed-size vector — an information bottleneck.
For machine translation, this causes quality degradation on long sentences.

**Attention mechanism** (Bahdanau, 2015) solves this by:
1. Keeping **all** encoder hidden states (not just the last one)
2. At each decoder step, computing a **weighted sum** over all encoder states
3. The weights are learned — the model learns which source words to focus on for each output word

This lets models handle arbitrarily long sequences without compression loss.

---

### 6.4 — Why Transformers Are Important in Modern NLP and Generative AI

Transformers (Vaswani et al., 2017) take attention further:

1. **No recurrence** — processes all tokens in parallel using Multi-Head Self-Attention
2. Each word attends to every other word simultaneously
3. Dramatically faster training due to full parallelism
4. Scales to billions of parameters without degradation

**Why this matters for Generative AI:**
- **GPT** uses transformer decoders to generate text autoregressively
- **BERT** uses bidirectional transformer encoders for classification and NER
- **Modern LLMs (Claude, GPT-4, LLaMA, Gemini)** are all built entirely on transformer architectures

> **The progression:** RNN (baseline) -> LSTM (better memory) -> Attention (no bottleneck) -> Transformer (parallel, scalable)
> Each step addresses the core weakness of the previous one.

---
## Save Evaluation Results

In [ ]:
os.makedirs('results', exist_ok=True)

lr_r  = classification_report(y_test, y_pred_lr,   target_names=class_names, output_dict=True)
nb_r  = classification_report(y_test, y_pred_nb,   target_names=class_names, output_dict=True)
lm_r  = classification_report(y_test, y_pred_lstm, target_names=class_names, output_dict=True)

rows = []
for label in class_names:
    rows.append({
        'Class'        : label,
        'LR_Precision' : round(lr_r[label]['precision'],3),
        'LR_Recall'    : round(lr_r[label]['recall'],3),
        'LR_F1'        : round(lr_r[label]['f1-score'],3),
        'NB_Precision' : round(nb_r[label]['precision'],3),
        'NB_Recall'    : round(nb_r[label]['recall'],3),
        'NB_F1'        : round(nb_r[label]['f1-score'],3),
        'LSTM_Precision': round(lm_r[label]['precision'],3),
        'LSTM_Recall'   : round(lm_r[label]['recall'],3),
        'LSTM_F1'       : round(lm_r[label]['f1-score'],3),
    })

eval_df = pd.DataFrame(rows)
eval_df.to_csv('results/model_evaluation.csv', index=False)
print('Saved -> results/model_evaluation.csv')
print(eval_df.to_string(index=False))

In [ ]:
with open('results/sample_predictions.txt', 'w') as f:
    f.write('SAMPLE PREDICTIONS - LSTM MODEL\n' + '='*60 + '\n\n')
    for i in range(15):
        true_lbl = label_map[y_test[i]]
        pred_lbl = label_map[y_pred_lstm[i]]
        correct  = 'CORRECT' if true_lbl == pred_lbl else 'WRONG'
        f.write(f'[{correct}] Sample {i+1}\n')
        f.write(f'  Original  : {X_test_list[i]}\n')
        f.write(f'  True      : {true_lbl}\n')
        f.write(f'  Predicted : {pred_lbl}\n\n')

print('Saved -> results/sample_predictions.txt')
print('\n' + '='*55)
print('PROJECT COMPLETE - All outputs saved to results/')
print('='*55)